<a href="https://colab.research.google.com/github/rymarinelli/Advanced-Programming-/blob/main/News_QA_Dataset.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install newsapi-python pandas tqdm
!pip install transformers torch accelerate sentencepiece
!pip install dspy-ai
!pip install newspaper3k "lxml[html_clean]"
# optional but helpful
!pip install bitsandbytes
!pip install datasets huggingface_hub

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 312.4/312.4 kB 20.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 139.7/139.7 kB 18.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.0/40.0 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.6/14.6 MB 134.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 413.9/413.9 kB 43.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.1/278.1 kB 37.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.4/7.4 MB 118.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.1/211.1 kB 28.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.5/81.5 kB 11.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [2]:
import os
from newsapi import NewsApiClient
import pandas as pd
from datetime import datetime, timedelta, timezone
from google.colab import userdata


NEWS_API_KEY = userdata.get('NEWS_API')
newsapi = NewsApiClient(api_key=NEWS_API_KEY)

def fetch_no_articles(query="Norge OR norsk", days_back=1, page_size=1, max_pages=2, language="no"):
    # NewsAPI "everything" supports language filtering (incl. 'no') depending on plan.
    from_date = (datetime.now(timezone.utc) - timedelta(days=days_back)).date().isoformat()
    all_rows = []
    for page in range(1, max_pages + 1):
        res = newsapi.get_everything(
            q=query,
            from_param=from_date,
            language=language,
            sort_by="publishedAt",
            page=page,
            page_size=page_size,
        )
        for a in res.get("articles", []):
            all_rows.append({
                "source": (a.get("source") or {}).get("name"),
                "source_id": (a.get("source") or {}).get("id"),
                "author": a.get("author"),
                "title": a.get("title"),
                "description": a.get("description"),
                "url": a.get("url"),
                "published_at": a.get("publishedAt"),
                # NewsAPI content is often truncated; still useful for dataset bootstrapping.
                "content_raw": a.get("content"),
            })
    return pd.DataFrame(all_rows)

df = fetch_no_articles(query="Norge OR norsk", days_back=14, max_pages=10)

In [3]:
from newspaper import Article

def extract_full_text(url):
    try:
        article = Article(url)
        article.download()
        article.parse()
        return article.text
    except:
        return None

df["full_text"] = df["url"].apply(extract_full_text)

In [4]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_QA = "norallm/normistral-11b-long"  # consider a smaller/distilled variant if needed
tokenizer_qa = AutoTokenizer.from_pretrained("norallm/normistral-11b-long")
model_qa = AutoModelForCausalLM.from_pretrained(
    MODEL_QA,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto",
)

config.json:   0%|          | 0.00/658 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/811 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/363 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

In [5]:
tokenizer_qa.chat_template = """{% for message in messages %}
{% if message['role'] == 'system' %}
{{ message['content'] }}

{% elif message['role'] == 'user' %}
[INST] {{ message['content'] }} [/INST]

{% elif message['role'] == 'assistant' %}
{{ message['content'] }}
{% endif %}
{% endfor %}"""


In [6]:
def deepseek_generate_qa(title, text):

    messages = [
        {"role": "system", "content": "Du lager norsk RAG-datasett og svarer kun i gyldig JSON."},
        {"role": "user", "content": f"""
Lag 1 spørsmål og 1 svar basert på artikkelen.

Krav:
- Norsk bokmål
- Kort svar (1–3 setninger)
- Kun info fra teksten
- STRICT JSON:
{{"question_no": "...", "answer_no": "..."}}

TITTEL: {title}
TEKST: {text}
"""}
    ]

    inputs = tokenizer_qa.apply_chat_template(
        messages,
        return_tensors="pt"
    )

    # 🔴 CRITICAL FIX
    inputs = {k: v.to(model_qa.device) for k, v in inputs.items()}

    with torch.no_grad():
        output = model_qa.generate(
            **inputs,
            max_new_tokens=200,
            do_sample=False,
            temperature=0.0,
            pad_token_id=tokenizer_qa.eos_token_id,
        )

    decoded = tokenizer_qa.decode(output[0], skip_special_tokens=True)
    return decoded


In [7]:
import dspy
import torch

class HFLocalModel(dspy.LM):
    def __init__(self, model, tokenizer):
        super().__init__('local-hf')
        self.model_binary = model
        self.tokenizer = tokenizer
        self.device = next(model.parameters()).device
        self.kwargs = {
            "temperature": 0.0,
            "max_tokens": 500
        }

    def __call__(self, prompt=None, messages=None, **kwargs):
        # Use tokenizer chat template if messages are provided
        if messages is not None:
            text_prompt = self.tokenizer.apply_chat_template(
                messages,
                tokenize=False,
                add_generation_prompt=True
            )
        else:
            text_prompt = prompt if prompt else ""

        # Tokenize and move to device
        inputs = self.tokenizer(text_prompt, return_tensors="pt").to(self.device)

        # Generate
        with torch.no_grad():
            output = self.model_binary.generate(
                **inputs,
                max_new_tokens=200,
                do_sample=False,
                pad_token_id=self.tokenizer.eos_token_id,
            )

        # Decode output
        # We only want the new tokens, but simple decode often includes input if not careful.
        # However, standard generate returns input+output.
        # Let's decode everything and split or just return as is depending on DSPy needs.
        # DSPy usually expects the full completion or just the new text depending on usage.
        # For many local HF implementations, returning the full text is safer if we don't slice strictly.
        # But usually we slice off the input length.

        # Slicing input length for cleaner output if possible
        input_len = inputs.input_ids.shape[1]
        generated_ids = output[0][input_len:]
        decoded = self.tokenizer.decode(generated_ids, skip_special_tokens=True)

        # Return as a list of strings
        return [decoded]

**Reasoning**:
Define the NewsDataset signature, instantiate the corrected HFLocalModel with the loaded model and tokenizer, configure dspy to use this local model, and generate a prediction for the first article to verify the setup.



In [9]:
class NewsDataset(dspy.Signature):
    """Generer strukturerte spørsmål fra en norsk nyhetsartikkel.
    Du må kun bruke informasjon som finnes direkte i teksten.
    Siter eksakt tekst der svaret finnes.
    """

    article = dspy.InputField(desc="Full norsk nyhetsartikkel")

    factual_question = dspy.OutputField(
        desc="Faktabasert spørsmål som kan besvares direkte fra teksten"
    )

    factual_answer = dspy.OutputField(
        desc="Kort svar (1–2 setninger) kun basert på informasjon i teksten"
    )

    answer_evidence = dspy.OutputField(
        desc="Eksakt sitat (ordrett) fra artikkelen som inneholder svaret. Må være direkte kopiert tekst."
    )

    context_question = dspy.OutputField(
        desc="Spørsmål som setter artikkelen i en bredere samfunns- eller nyhetskontekst"
    )

    perspective_question = dspy.OutputField(
        desc="Spørsmål om perspektiv, implikasjoner eller mulig tolkning av saken"
    )

    political_leaning = dspy.OutputField(
        desc=(
            "Vurder artikkelens politiske orientering på en kontinuerlig skala fra 0 til 1. "
            "0 = klart konservativ eller ikke-liberal framing, "
            "0.5 = nøytral eller balansert journalistisk fremstilling, "
            "1 = klart liberal/progressiv framing. "
            "Basér vurderingen kun på språk, vinkling og implisitte perspektiver i teksten."
        )
    )


# Instantiate the local model wrapper
lm = HFLocalModel(model_qa, tokenizer_qa)

# Configure dspy
dspy.settings.configure(lm=lm)

# Create the predictor
generator = dspy.Predict(NewsDataset, adapter=dspy.JSONAdapter())

# Prepare input
# Handle potential missing content gracefully
title = df.iloc[0]["title"] or ""
content = df.iloc[0]["full_text"] or ""
test_article = f"{title}\n{content}"

# Run prediction
print(f"Processing article: {title[:50]}...")
pred = generator(article=test_article)

# Display result
print("\n--- Resultat ---")
print(f"Factual Question: {pred.factual_question}")
print(f"Factual Answer: {pred.factual_answer}")
print(f"Answer Evidence: {pred.answer_evidence}")
print(f"Perspective Question: {pred.perspective_question}")
print(f"Political Landscape:{pred.political_leaning}")

Processing article: USA trekker kampfly fra Nato-øvelse i Nord-Norge: ...


AdapterParseError: LM response cannot be serialized to a JSON object.

Adapter JSONAdapter failed to parse the LM response. 

LM Response: ### Example

[[ ## article ## ]]
USA trekker kampfly fra Nato-øvelse i Nord-Norge: – Iran-situasjonen endrer planene
USA trekker store deler av sine planlagte luftstyrker, inkludert moderne F-35 kampfly, fra øvelsen Cold Response.

Lytt til saken

Forsvaret bekrefter overfor Fremover at de i dag har fått beskjed om at enkelte amerikanske styrker ikke kommer til å delta.

– De skal andre steder i verden, sier oberstløytnant Espen Solemdal overfor Fremover.

Les også: Trump: «Vi vil være klare»

Nato-øvelsen Cold Response skal samle tusenvis av soldater i Norge og Norden. Mens noen Osprey-enheter ifølge avisa fortsatt ventes, skal hovedstyrken av F-35-fly ikke delta. Det betyr at rundt 100–150 amerikanske soldater ikke kommer til Norge.

Årsaken er den militære oppbyggingen USA gjør i Midtøsten, hvor de forbereder seg på en potensiell stor 

Expected 

In [ ]:
import pandas as pd
from datasets import Dataset
from tqdm import tqdm
from google.colab import userdata

# Initialize lists to store generated data
results = {
    "factual_question": [],
    "factual_answer": [],
    "answer_evidence": [],
    "context_question": [],
    "perspective_question": [],
    "political_leaning": []
}

print(f"Processing {len(df)} articles...")

# Iterate over the dataframe
for index, row in tqdm(df.iterrows(), total=len(df)):
    try:
        # Prepare input text
        title = row.get("title") or ""
        content = row.get("content_raw") or ""
        article_text = f"{title}\n{content}"

        # Generate prediction
        pred = generator(article=article_text)

        # Store results
        results["factual_question"].append(pred.factual_question)
        results["factual_answer"].append(pred.factual_answer)
        results["answer_evidence"].append(pred.answer_evidence)
        results["context_question"].append(pred.context_question)
        results["perspective_question"].append(pred.perspective_question)
        results["political_leaning"].append(pred.political_leaning)

    except Exception as e:
        print(f"Error processing row {index}: {e}")
        # Append None or empty strings to maintain alignment
        for key in results:
            results[key].append(None)

# Add generated columns to the dataframe
for key, values in results.items():
    df[key] = values


# Push to Hub
hf_token = userdata.get("HF_TOKEN")
hf_dataset.push_to_hub("norwegian-news-qa", token=hf_token)

print("Dataset successfully pushed to Hugging Face Hub!")